# Session 1 – Introduction to Unsupervised Learning

## 1. Random Dataset of 10 App Users
Each user has two features:
- **daily_app_opens** – how many times the user opens the app per day
- **avg_session_duration** – average session length in minutes

In [1]:
import random

random.seed(42)  # for reproducible results

users = []
for i in range(1, 11):
    users.append({
        "user_id": i,
        "daily_app_opens": random.randint(1, 30),
        "avg_session_duration": round(random.uniform(1, 60), 2)  # minutes
    })

print(users)
print()
for u in users:
    print(u)

[{'user_id': 1, 'daily_app_opens': 21, 'avg_session_duration': 7.57}, {'user_id': 2, 'daily_app_opens': 24, 'avg_session_duration': 17.23}, {'user_id': 3, 'daily_app_opens': 8, 'avg_session_duration': 9.23}, {'user_id': 4, 'daily_app_opens': 4, 'avg_session_duration': 40.93}, {'user_id': 5, 'daily_app_opens': 29, 'avg_session_duration': 33.18}, {'user_id': 6, 'daily_app_opens': 19, 'avg_session_duration': 25.89}, {'user_id': 7, 'daily_app_opens': 1, 'avg_session_duration': 6.53}, {'user_id': 8, 'daily_app_opens': 8, 'avg_session_duration': 30.82}, {'user_id': 9, 'daily_app_opens': 1, 'avg_session_duration': 34.11}, {'user_id': 10, 'daily_app_opens': 23, 'avg_session_duration': 39.34}]

{'user_id': 1, 'daily_app_opens': 21, 'avg_session_duration': 7.57}
{'user_id': 2, 'daily_app_opens': 24, 'avg_session_duration': 17.23}
{'user_id': 3, 'daily_app_opens': 8, 'avg_session_duration': 9.23}
{'user_id': 4, 'daily_app_opens': 4, 'avg_session_duration': 40.93}
{'user_id': 5, 'daily_app_opens':

## 2. Supervised vs Unsupervised Learning

| # | Point of Comparison | Supervised Learning | Unsupervised Learning |
|---|---|---|---|
| 1 | **Definition** | Learns a mapping from inputs to a *known* output using labeled data. | Finds hidden structure/patterns in data *without* labels. |
| 2 | **Type of Data** | Labeled data (input + correct answer). <br>*Instagram feed ranking:* past posts labeled as liked / not liked, watched time, etc. | Unlabeled data (only inputs). <br>*Spotify playlist grouping:* song audio features (tempo, energy, danceability) with no genre tag given. |
| 3 | **Goal** | Predict an outcome for new data. <br>*Gmail:* predict whether an email is **spam or not spam**. | Discover groups, associations or structure. <br>*Amazon / Flipkart:* "Frequently bought together" via association rules. |
| 4 | **Common Tasks** | Classification & Regression. <br>*Uber / Ola:* predicting ride fare (regression); *Instagram:* predicting if a user will like a post (classification). | Clustering, Dimensionality Reduction, Association, Anomaly Detection. <br>*Spotify:* grouping similar songs/listeners for "Daily Mix"; *Paytm:* detecting unusual transactions. |
| 5 | **Common Algorithms** | Linear/Logistic Regression, Decision Trees, Random Forest, SVM, Neural Nets. | K-Means, Hierarchical Clustering, DBSCAN, PCA, Apriori. |
| 6 | **Evaluation** | Easy to measure – compare predictions with true labels (accuracy, precision, RMSE). | Harder – no ground truth; use Silhouette score, inertia (elbow), or business judgement. |
| 7 | **Human Effort** | High – someone must label data (e.g., YouTube content moderators tagging videos). | Low – raw data can be used directly (e.g., Netflix viewing logs). |
| 8 | **Output** | A specific label or number (e.g., "this post ranks #3 in your feed"). | Groups / patterns that need interpretation (e.g., "Cluster 2 = late-night lo-fi listeners"). |

## 3. Segmenting 8 Products into Clusters
We have 8 products with **price (₹)** and **number of reviews** (a proxy for popularity).
First we group them manually, then confirm the grouping using K-Means.

In [2]:
products = [
    {"product": "USB Cable",         "price": 199,   "reviews": 5200},
    {"product": "Phone Case",        "price": 299,   "reviews": 4800},
    {"product": "Wired Earphones",   "price": 499,   "reviews": 6100},
    {"product": "Fitness Band",      "price": 1999,  "reviews": 2100},
    {"product": "Bluetooth Speaker", "price": 2499,  "reviews": 1800},
    {"product": "Smartwatch",        "price": 2999,  "reviews": 1500},
    {"product": "DSLR Camera",       "price": 55000, "reviews": 220},
    {"product": "Laptop",            "price": 65000, "reviews": 350},
]

# ---- Manual (rule-based) grouping ----
def manual_cluster(p):
    if p["price"] < 1000 and p["reviews"] > 4000:
        return "Cluster A - Budget & Very Popular"
    elif p["price"] < 10000:
        return "Cluster B - Mid-Range & Moderately Popular"
    else:
        return "Cluster C - Premium & Niche"

for p in products:
    p["manual_cluster"] = manual_cluster(p)
    print(f"{p['product']:<18} Rs.{p['price']:>6}  reviews={p['reviews']:>5}  ->  {p['manual_cluster']}")

USB Cable          Rs.   199  reviews= 5200  ->  Cluster A - Budget & Very Popular
Phone Case         Rs.   299  reviews= 4800  ->  Cluster A - Budget & Very Popular
Wired Earphones    Rs.   499  reviews= 6100  ->  Cluster A - Budget & Very Popular
Fitness Band       Rs.  1999  reviews= 2100  ->  Cluster B - Mid-Range & Moderately Popular
Bluetooth Speaker  Rs.  2499  reviews= 1800  ->  Cluster B - Mid-Range & Moderately Popular
Smartwatch         Rs.  2999  reviews= 1500  ->  Cluster B - Mid-Range & Moderately Popular
DSLR Camera        Rs. 55000  reviews=  220  ->  Cluster C - Premium & Niche
Laptop             Rs. 65000  reviews=  350  ->  Cluster C - Premium & Niche


In [3]:
# ---- Verify with K-Means (features scaled because price and reviews have very different ranges) ----
try:
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans

    X = np.array([[p["price"], p["reviews"]] for p in products])
    X_scaled = StandardScaler().fit_transform(X)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    for p, lbl in zip(products, labels):
        print(f"{p['product']:<18} -> KMeans cluster {lbl}")
except ImportError:
    print("scikit-learn not installed - run `pip install scikit-learn` to verify with K-Means.")

USB Cable          -> KMeans cluster 0
Phone Case         -> KMeans cluster 0
Wired Earphones    -> KMeans cluster 0
Fitness Band       -> KMeans cluster 2
Bluetooth Speaker  -> KMeans cluster 2
Smartwatch         -> KMeans cluster 2
DSLR Camera        -> KMeans cluster 1
Laptop             -> KMeans cluster 1


### Reasoning for each cluster

| Cluster | Products | Price Range | Reviews | Reasoning |
|---|---|---|---|---|
| **A – Budget & Very Popular** | USB Cable, Phone Case, Wired Earphones | ₹199 – ₹499 | 4,800 – 6,100 | Cheap, everyday accessories. Low price → impulse buys by a huge audience → very high number of reviews. |
| **B – Mid-Range & Moderately Popular** | Fitness Band, Bluetooth Speaker, Smartwatch | ₹1,999 – ₹2,999 | 1,500 – 2,100 | Lifestyle gadgets – a considered purchase but still affordable. Moderate price and moderate popularity. |
| **C – Premium & Niche** | DSLR Camera, Laptop | ₹55,000 – ₹65,000 | 220 – 350 | Expensive, high-involvement purchases bought by fewer customers → few reviews. |

**Key observation:** Price and reviews are *inversely related* here – as price rises, the number of buyers (and reviews) falls.
So products that are close in **both** dimensions naturally fall into the same group.
A business could use these segments to e.g. bundle Cluster A items as add-ons, run EMI offers on Cluster C, and target fitness/lifestyle ads for Cluster B.

## 4. Unsupervised Learning Scenarios in Zomato, Flipkart & Spotify

| # | App & Scenario | Data Needed | Possible Insights / Hidden Patterns |
|---|---|---|---|
| 1 | **Zomato – Customer Segmentation (Clustering)** | Order frequency, average order value, preferred cuisines, order time (lunch/late-night), discount/coupon usage, location. | Discover groups like *"late-night snackers"*, *"weekend family orderers"*, *"discount hunters"*, *"premium diners"*. Zomato can send targeted offers (e.g., Zomato Gold to premium diners, coupons to price-sensitive users). |
| 2 | **Flipkart – Market Basket Analysis (Association Rules / Apriori)** | Transaction history: which products are bought together in the same cart/session, timestamps, categories. | Hidden rules like *"Phone → Screen guard + Back cover"* or *"Baby diapers → Wet wipes"*. Used for "Frequently Bought Together", combo offers, and better warehouse placement. Also clustering products for similar-item recommendations. |
| 3 | **Spotify – Song & Listener Grouping (Clustering)** | Audio features of songs (tempo, energy, danceability, acousticness, valence), listening history, skip rate, time of day, playlist co-occurrence. | Group songs into *moods* (chill, workout, focus) without manual tags, and group listeners with similar taste. Powers **Daily Mix**, **Discover Weekly** and mood playlists; reveals emerging micro-genres. |

**Bonus:** *Flipkart / Paytm – Anomaly Detection*: using order amount, location, device and return patterns to spot unusual behaviour such as fake reviews, fraudulent returns or bot accounts – without having pre-labelled fraud examples.

## 5. Real-World Dataset for Customer Segmentation

### Mall Customer Segmentation Data (Kaggle)
🔗 **Link:** https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
(file: `Mall_Customers.csv`)

**Structure**
- **Rows:** 200 customers
- **Columns:** 5 (1 ID + 4 features)

| Column | Data Type | Kind of Data | Description |
|---|---|---|---|
| `CustomerID` | Integer | Identifier | Unique customer ID (not used for modelling) |
| `Gender` | String | Categorical (Male/Female) | Gender of the customer |
| `Age` | Integer | Numerical (continuous) | Age in years (18 – 70) |
| `Annual Income (k$)` | Integer | Numerical (continuous) | Yearly income in thousand dollars (15 – 137) |
| `Spending Score (1-100)` | Integer | Numerical | Score assigned by the mall based on spending behaviour |

**Labeled or Unlabeled?** – **Unlabeled.** There is no target column telling which segment a customer belongs to,
so it is perfect for unsupervised learning (e.g., K-Means on *Annual Income* vs *Spending Score* typically reveals ~5 segments such as
"high income – low spenders", "low income – high spenders", etc.).

---

### Alternative: Online Retail Dataset (UCI ML Repository)
🔗 **Link:** https://archive.ics.uci.edu/dataset/352/online+retail

- **Rows:** 541,909 transactions from a UK-based online gift retailer (Dec 2010 – Dec 2011)
- **Columns:** 8 – `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`
- **Data types:** mix of categorical (codes, description, country), numerical (quantity, price) and date-time
- **Labeled or Unlabeled?** – **Unlabeled.** Commonly used to build **RFM (Recency, Frequency, Monetary)** features per customer and then cluster customers.

In [5]:
# Optional: load the Mall Customers dataset (download Mall_Customers.csv from the Kaggle link above)
import pandas as pd

try:
    df = pd.read_csv("Mall_Customers.csv")
    print("Shape:", df.shape)
    print(df.head())
    print(df.dtypes)
except FileNotFoundError:
    print("Mall_Customers.csv not found - download it from Kaggle and place it next to this notebook.")

Shape: (200, 5)
   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40
CustomerID                int64
Gender                      str
Age                       int64
Annual Income (k$)        int64
Spending Score (1-100)    int64
dtype: object
